# Week 10 Problem Set: The List

You are the field director. Nine thousand doors, 60,000 registered voters, and a vendor file you did not build.

Four tasks. **Graded pass/fail**, like every problem set in this course: attempt all of it with real effort and you have a pass.

**Lying with data, the checklist so far:**
1. **W1:** Conflating fixed and marginal costs.
2. **W2:** Presenting an observational comparison as a causal effect.
3. **W3:** Applying a result from one setting to a different one.
4. **W4:** Cherry-picking the winning arm from a multi-arm test.
5. **W5:** Treating an underpowered null as evidence of no effect.
6. **W7:** Reporting the complier comparison as a causal effect.
7. **W8:** Cherry-picking polls; house effects; ignoring nonresponse bias.
8. **W9:** Comparing by effect size without cost.
9. **W10:** An accuracy number quoted with no floor beside it, a model graded on the data it was built from, and a model of **who votes** sold as a model of **who you can move**.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the voter file straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

We split the file the same way we did in class. Build on a random 42,000, and grade on the 18,000 the model has never seen. That is the only way to find out whether a model learned a pattern or memorized some people.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

vf = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk10_turnout_models/data/district_voter_file.csv')
train = vf.sample(frac=0.7, random_state=10)
test = vf.drop(train.index).copy()
print(len(vf), 'registered voters;', len(train), 'to build on,', len(test), 'held back')

## Task 1: The floor

Keystone's headline claim is that their model classified 80.0% of registered voters correctly against the 2024 general election.

An accuracy number means nothing without the score of the dumbest possible model beside it. The **floor** is what you get by writing the *same* answer on every row, whichever answer is more common. When turnout is above half that is "will vote," and the floor is the turnout rate. When turnout is below half it is "will not vote," and the floor is one minus the turnout rate.

Fill in the two blanks.

*Check:* the 2024 floor is 75.9% and the 2022 floor is 50.9%.

In [ ]:
keystone_call = (test['keystone_score'] >= 0.5).astype(int)
acc_2024 = (keystone_call == test['voted_2024']).mean()

t24 = test['voted_2024'].mean()
t22 = test['voted_2022'].mean()

# The floor is whichever single answer is right more often, so it is
# max(the turnout rate, one minus the turnout rate).
floor_2024 = # YOUR CODE HERE

floor_2022 = # YOUR CODE HERE

print('Keystone vs 2024: accuracy', round(100 * acc_2024, 1),
      '% against a floor of', round(100 * floor_2024, 1), '%')
print('2022 floor:', round(100 * floor_2022, 1), '%')

**Question 1.** In two sentences: how much work is Keystone's model doing against 2024, and why does that number look so much better than it is?

**Question 2.** Turnout on this file in 2014 was 41.1%. What is the floor for 2014, and which answer gets it?

*Your answers here.*

## Task 2: Which mistakes

Build the midterm model, then count its four kinds of answer by hand.

*Check:* TP 7,526, FP 1,717, FN 1,645, TN 7,112. They sum to 18,000, and accuracy is 81.3%, precision 81.4%, recall 82.1%.

In [ ]:
X = 'voted_2012 + voted_2014 + voted_2016 + voted_2018 + age'
mid = smf.ols('voted_2022 ~ ' + X, data=train).fit()
test['our_score'] = mid.predict(test)
our_call = (test['our_score'] >= 0.5).astype(int)

# Count each of the four boxes. tp is done for you; write the other three.
tp = ((our_call == 1) & (test['voted_2022'] == 1)).sum()
fp = # YOUR CODE HERE
fn = # YOUR CODE HERE
tn = # YOUR CODE HERE

print('TP', tp, ' FP', fp, ' FN', fn, ' TN', tn, ' total', tp + fp + fn + tn)
print('accuracy ', round(100 * (tp + tn) / len(test), 1), '%')
print('precision', round(100 * tp / (tp + fp), 1), '%')
print('recall   ', round(100 * tp / (tp + fn), 1), '%')

**Question 3.** Keystone's recall against 2024 is 95.2%, better than our model's 82.1%. Explain in three sentences why that does not make Keystone the better model. What would a model's recall be if it called every single person a voter?

*Your answer here.*

## Task 3: Calibration, and where the model breaks

Accuracy throws the number away and keeps a yes or a no. **Calibration** keeps the number: of the people the model scores 0.60, do about 60% vote?

The helper below is the one from live coding. Run it on our midterm model, and then on Keystone's.

*Check:* the worst bin is the bottom one. The model gives those 1,466 people an average of 8.5% and 4.2% of them voted, a gap of 4.2 points. Every other bin is inside 4.

In [ ]:
def calibration(scores, actual):
    top = max(1.0, np.ceil(scores.max() * 10) / 10)
    bins = pd.cut(scores, np.arange(0, top + 0.001, 0.1), include_lowest=True)
    out = pd.DataFrame({'score': scores, 'voted': actual}).groupby(bins, observed=True).agg(
        n=('voted', 'size'), model_said=('score', 'mean'), actually_voted=('voted', 'mean'))
    out['gap_pts'] = (out['model_said'] - out['actually_voted']) * 100
    return out.round(3)

calibration(test['our_score'], test['voted_2022'])

In [ ]:
calibration(test['keystone_score'], test['voted_2022'])

**Question 4.** Compare the two tables. Read the `gap_pts` column of Keystone's and say in one sentence what is wrong with it, and whether subtracting a fixed number from every score would fix it.

*Your answer here.*

Well calibrated *on average* does not mean right about everybody. Break the same comparison out by age.

The cell below does the same thing as the `calibration` helper you just ran: group, aggregate three columns, then add a gap column. One difference: the helper had to build a DataFrame first because it was handed two loose columns, and you already have `test`, so you can go straight to `test.groupby(...)`.

*Check:* the youngest band has n = 1,661, the model says 12.2%, and 23.9% actually voted. That is a gap of **−11.7 points**, and it is the worst band by a distance.

In [ ]:
bands = pd.cut(test['age'], [19, 25, 34, 44, 54, 64, 91])

# Group `test` by `bands` and report, for each band: n, the mean of 'our_score',
# and the mean of 'voted_2022'. Then add a gap column in percentage points.
by_age = # YOUR CODE HERE

by_age

**Question 5.** Name the band the model gets worst and say which direction it is wrong in.

Then explain **why**, using the voter file itself. The two cells below are the evidence. Run them before you answer.

In [ ]:
# How many people on the file were too young to legally vote in each election?
for year, needed_age in [(2012, 32), (2014, 30), (2016, 28), (2018, 26)]:
    n = (vf['age'] < needed_age).sum()
    print('voted_' + str(year), ':', n, 'people could not legally vote',
          '(' + str(round(100 * n / len(vf), 1)) + '% of the file)')

In [ ]:
# Everyone who voted in none of the four elections the model uses, split by age
never = vf[(vf['voted_2012'] == 0) & (vf['voted_2014'] == 0)
           & (vf['voted_2016'] == 0) & (vf['voted_2018'] == 0)]
young = never[never['age'] < 26]
older = never[never['age'] >= 26]

print('people with four zeros:', len(never))
print('  under 26   :', len(young), 'of whom',
      round(100 * young['voted_2022'].mean(), 1), '% voted in 2022')
print('  26 and over:', len(older), 'of whom',
      round(100 * older['voted_2022'].mean(), 1), '% voted in 2022')

*Your answer here.*

## Task 4: The memo

Run the cell below, then write **250–350 words** to Rachel, your campaign manager. She has ten minutes and a printer deadline.

In [ ]:
vf['score'] = mid.predict(vf)
vf['band'] = pd.cut(vf['score'], [-1, 0.35, 0.65, 2], labels=['low', 'middle', 'high'])

for name, rows in [('9,000 highest', vf.nlargest(9000, 'score')),
                   ('9,000 middle ', vf[vf['band'] == 'middle'].sample(9000, random_state=3)),
                   ('9,000 lowest ', vf.nsmallest(9000, 'score'))]:
    print(name, '| mean score', format(rows['score'].mean(), '.2f'),
          '| voted in 2022:', str(round(100 * rows['voted_2022'].mean(), 1)) + '%')

print()
print(pd.crosstab(vf['band'], vf['party']))

**Question 6.** Katie says the middle band is the only place a knock changes an outcome. Is that something you can check in this voter file? Answer in two sentences, and say what evidence would settle it.

*Your answer here.*

Your memo must:

1. **Open with the recommendation.** Which 9,000 doors. Say it in the first sentence.
2. **Give your reasons**, using at least two numbers you computed above.
3. **Steelman the strongest objection to your own recommendation**, and answer it. Marcus and Rachel both have real arguments; pick whichever one cuts hardest against you.
4. **Name what would change your mind.** One number, where you would get it, and which way it would have to come out.

I am grading whether the memo is finished, not whether I agree with it. A memo with no counter-argument is not finished.

*Your memo here.*

---

**Due at 4:00pm on Wednesday Nov 18**, to the **problem set** assignment on Canvas. Whatever you had at 5:55pm in class already went to the separate **in-class** assignment; that one is your attendance credit and you do not resubmit it.


## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished. Fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Open the PDF and read it before you upload.** The PDF will look complete even when it isn't. Every heading and prompt prints whether or not the code ran. What matters is the **output**: under each code cell you should see a table, a number, or a plot. A red error box, or `In [ ]` with nothing beneath it, means that part did not run and will be graded as missing. Also check that your memo printed in full.
5. Upload the PDF to Canvas.